In [1]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font

# File paths
FY = "_fy25"
input_dir = '../data/extracts/'
output_dir = '../data/outputs/'
csv_file = input_dir + 'building_complex_sf' + FY + '.csv'
excel_file = input_dir + 'building_complex_master_sheet' + FY + '.xlsx'
sheet_name = "complex"
target_col_idx = 14  # Target column index (Column O = 15th column → index 14)
report_file = output_dir + 'sf_report.xlsx'

# Load CSV
df_csv = pd.read_csv(csv_file)

# Ensure counts are integers and IDs are strings
df_csv['count bldgs'] = df_csv['count bldgs'].fillna(0).astype(int)
df_csv['count sf'] = df_csv['count sf'].fillna(0).astype(int)
df_csv['building_complex_id'] = df_csv['building_complex_id'].astype(str).str.strip()

# Separate rows to insert vs skipped
df_to_insert = df_csv[df_csv['count bldgs'] == df_csv['count sf']]
df_skipped = df_csv[df_csv['count bldgs'] != df_csv['count sf']]

# Load Excel
wb = load_workbook(excel_file)
ws = wb[sheet_name]  # use specific sheet name instead of active

# Map Excel IDs (as string) to row objects, skipping 2 header rows
excel_id_to_row = {str(r[1].value).strip(): r for r in ws.iter_rows(min_row=3, max_row=ws.max_row)}

# Track missing IDs
missing_ids = []

# Update Excel
for _, r in df_to_insert.iterrows():
    csv_id = r['building_complex_id']
    if csv_id in excel_id_to_row:
        excel_row = excel_id_to_row[csv_id]
        excel_row[target_col_idx].value = r['sum']  # Column O
    else:
        missing_ids.append(csv_id)

# Save updated Excel
wb.save(excel_file)



## Prepare report sheets: report buildings with 'count bldgs' != 'count sf' ##

output_sheets = {}
if not df_skipped.empty:
    output_sheets['Skipped_Count_Mismatch'] = df_skipped[['building_complex_id', 
                                                          'building_complex_name', 
                                                          'count bldgs', 
                                                          'count sf']]
if missing_ids:
    output_sheets['IDs_Not_Found'] = pd.DataFrame({'building_complex_id': missing_ids})

# Write report with formatting
if output_sheets:
    with pd.ExcelWriter(report_file, engine='openpyxl') as writer:
        for sheet_name, df in output_sheets.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            ws_report = writer.sheets[sheet_name]
            # Apply font 12 and auto-adjust column width
            for col_idx, col in enumerate(df.columns, 1):
                max_length = max(df[col].astype(str).map(len).max(), len(col))
                col_letter = get_column_letter(col_idx)
                ws_report.column_dimensions[col_letter].width = max_length + 2  # small padding
                for cell in ws_report[col_letter]:
                    cell.font = Font(size=12)

    print(f"Report saved to {report_file}")
else:
    print("No skipped rows or missing IDs to report.")

# Summary
print("\nUpdate Summary:")
print(f"Total CSV rows: {len(df_csv)}")
print(f"Successfully inserted: {len(df_to_insert) - len(missing_ids)}")
print(f"Skipped due to count mismatch: {len(df_skipped)}")
print(f"IDs not found in Excel: {len(missing_ids)}")


Report saved to ../data/outputs/sf_report.xlsx

Update Summary:
Total CSV rows: 188
Successfully inserted: 167
Skipped due to count mismatch: 19
IDs not found in Excel: 2


In [2]:
missing_ids

['1152', '1032']